<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 12 · Deep Learning for Cross-Sectional and Panel Data

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
We implement PyTorch MLPs for cross-sectional return prediction and autoencoder-based factor extraction.

感悟：Neural Network像是一种连续函数逼近机器⚙️。  



### Getting Help While Using PyTorch
- **Appendix D** provides PyTorch syntax and training tips.
- **Appendix B** covers the pandas transformations used for features.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

import torch
import torch.nn as nn   # PyTorch.nn像是策略模型结构生成器（模型样子）。比如：Linear, LSTM, Transformer, Conv(局部模式提取)。
from torch.utils.data import TensorDataset, DataLoader
# torch.utils 就是 PyTorch 给你准备的【工具箱】：
# 1. torch.utils.data 就是造数据集、批量喂数据；
# 2. torch.utils.checkpoint 是训练可视化；
# 3. torch.utils.checkpoint 是断点续训；
# 另，torch.utils.data.DataLoader 是动态的数据流，将大数据切成小批量，喂给神经网络训练。
#    torch.utils.data.dataset 是静态数据源。

### Data Loading
We load the price panel once for this notebook.

In [2]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index().ffill()

## 1. Prepare Training Data

We extract a simple feature/label table from the price panel and convert it into PyTorch tensors, ready for mini‑batch training.  
X, y（全部张量）  
     ↓  
切成前80%训练、后20%验证  
     ↓  
放进 TensorDataset（包装成对儿的数据）  
     ↓  
放进 DataLoader  
     ↓  
train_loader   每次输出：  
   batch_x = (512, 1)  
   batch_y = (512)

In [3]:
panel = prices.ffill().pct_change().dropna()


features = panel.rolling(20).mean().stack().dropna()
# stack()是将 DataFrame ——> Series, reindex()只改变行号，不改变数据类型。
labels = panel.shift(-1).stack().reindex(features.index).dropna()


features = features.loc[labels.index]
X = torch.from_numpy(features.values.astype(np.float32)).unsqueeze(-1)
# torch.from_numpy()数据从numpy变成了tensor.
# unsqueeze(-1)给数据强加一个维度，值永远是1.
y = torch.from_numpy(labels.values.astype(np.float32))


train_size = int(0.8 * len(X))
# dataframe变成 numpy/tensor后，index就没有了，但顺序还在。只能以这种方式划分train and test。
train_ds = TensorDataset(X[:train_size], y[:train_size])
val_ds = TensorDataset(X[train_size:], y[train_size:])


train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)  # 量化金融不可以shuffle，除非是一天的截面数据。
val_loader = DataLoader(val_ds, batch_size=512)

# for batch_x, batch_y in train_loader: （每次循环拿到的是batch_x:512个样本的特征；batch_y:512个样本的标签）
    # model training

## 2. Multi-Layer Perceptron

This section defines a compact MLP architecture with one hidden layer, ReLU activation, and dropout for regularization.

In [4]:
class MLP(nn.Module):             # 本例是 2-layer or 1-hidden layer model。
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),  # 第一层 从1到64.
            nn.ReLU(),             # 激活函数
            nn.Dropout(0.2),       # 随机失活
            nn.Linear(hidden, 1),  # 第二层，从64到1.
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)
        # self.net()输出shape:(batch_size, 1), 因此去掉最后一列（与unsqueeze(-1)相反）,shape变成了(batch_size,).
        # squeeze(-1)是把模型输出压扁，形状和label形状对齐，bu

model = MLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam更新的是w和b.
loss_fn = nn.MSELoss()    # 压扁后的预测结果，形状相同的label都聚集在此。

### 2.1 Training Loop

We implement a minimal training loop with batches, loss computation, backpropagation, and validation loss tracking.

In [5]:
def train(model, epochs=5):
    for epoch in range(epochs):

        model.train()                 # 现在进入学习模式：
        for xb, yb in train_loader:
            optimizer.zero_grad()     # 先把旧的梯度清空，因为PyTorch默认累加梯度。
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()           # 计算loss对参数的偏导数。
            optimizer.step()          # 修改model。

        model.eval()                  # 现在进入考试模式：此时dropout关闭。
        with torch.no_grad():         # 此时，只测试不学习，不记录梯度，no backward,节省内存，提高速度。
            val_losses = []
            for xb, yb in val_loader:
                val_losses.append(loss_fn(model(xb), yb).item())

                # .item() = 从张量里取出【唯一的那个数字】开始统计。"特种兵，把枪放下，我只想看你脸长啥样！"
                # 张量不能直接打印、不能画图、不能存日志，因为它是背着一堆 “秘密任务” 的超级工具！
                # 它不只是一个数字，它是带着任务、带着身份、带着装备的特种兵：
                # 它实际装了：
                  # 1）这个数字在 CPU 还是 GPU；
                  # 2）是否需要计算梯度（backpropagation）；
                  # 3）数据类型 float32 / float64；
                  # 4）数据形状 (512, 1)；
                  # 5）计算图的连接关系；
                  # 6）梯度值本身。

                # 在损失估计上PyTorch与sklearn是反的！
                  # PyTorch 是先算模型输出 → 再和真实值比。
                  # 所以它习惯: 模型输出 (pred) 放第 1 位，真实标签 (true) 放第 2 位。
                # sklearn 是纯评估工具 → 习惯：真实值在前，预测在后。
        print(f"Epoch {epoch+1}, val loss {np.mean(val_losses):.5f}")

train(model, epochs=5)

Epoch 1, val loss 0.00126
Epoch 2, val loss 0.00036
Epoch 3, val loss 0.00036
Epoch 4, val loss 0.00036
Epoch 5, val loss 0.00036


## 3. Autoencoder for Factor Extraction

Here we build an autoencoder whose bottleneck layer serves as a learned low‑dimensional factor representation of the inputs.  
流程： 输入 → 编码器（压缩）→ 隐向量 → 解码器（还原）→ 重构输出  

本质：重构自己。  

用途：原始粗糙因子 → AE 压缩提纯 → 隐向量（可用做选股因子）


In [6]:
class AutoEncoder(nn.Module):            # 自定义 AutoEncoder类，用积木拼出来的汽车🚗。
    def __init__(self, latent=3):
        super().__init__()

        # encoder 负责压缩特征
        self.encoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, latent),       # 输出维度 = latent
        )

        # decoder负责还原特征
        self.decoder = nn.Sequential(
            nn.Linear(latent, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )


    def forward(self, x):
        # 🎯 这里就是隐向量
        # 编码器输出 = latent vector
        latent = self.encoder(x)
        recon = self.decoder(latent)
        return latent, recon

auto = AutoEncoder()
opt = torch.optim.Adam(auto.parameters(), lr=1e-3)

### 3.1 Train Autoencoder

We train the autoencoder on the feature tensor and then inspect the latent codes that emerge from the encoder.  

【Autoencoder】目的是：提取提纯后的新因子。   
1.训练n轮；  
2.打开训练模式；  
3.遍历第一批数据；  
4.清空梯度；  
5.前向传播 --> 得到重构值；  
6.计算损失：还原得像不像？  
7.反向传播 --> 优化  
8.记录损失  


In [7]:
for epoch in range(5):
    auto.train()
    epoch_loss = 0
    for xb, _ in train_loader:    # xb为一批原始特征，_为用不到的标签y。
        opt.zero_grad()
        _, recon = auto(xb)       # auto(xb)输出（latent, recon), 而latent训练时暂时不用，写_.
        loss = loss_fn(recon.squeeze(-1), xb.squeeze(-1))
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}, loss {epoch_loss / len(train_loader):.5f}")

with torch.no_grad():
    latent_factors, _ = auto(X)   # 编码器输出的隐向量（提纯后的新因子）
latent_factors[:5]                # 3个独立的、提纯之后的新因子

Epoch 1, loss 0.00003
Epoch 2, loss 0.00003
Epoch 3, loss 0.00003
Epoch 4, loss 0.00003
Epoch 5, loss 0.00002


tensor([[-0.1446,  0.2727,  0.1896],
        [-0.1454,  0.2736,  0.1881],
        [-0.1451,  0.2733,  0.1887],
        [-0.1451,  0.2732,  0.1888],
        [-0.1451,  0.2732,  0.1888]])

In [9]:
# 把 tensor 转成 DataFrame，索引还是原来的 日期+股票
factor_df = pd.DataFrame(
    latent_factors.cpu().numpy(),
    index=features.index,
    columns=["factor_1", "factor_2", "factor_3"]
)
factor_df

factor_1  factor_2  factor_3
Date                                            
2015-12-29 AAPL    -0.144618  0.272742  0.189649
           NVDA    -0.145409  0.273593  0.188135
           JPM     -0.145113  0.273275  0.188701
           SPY     -0.145071  0.273230  0.188781
           GLD     -0.145087  0.273247  0.188751
...                      ...       ...       ...
2025-11-26 SPY     -0.145008  0.273162  0.188901
           GLD     -0.145368  0.273549  0.188213
           TLT     -0.145059  0.273217  0.188804
           EURUSD  -0.145065  0.273223  0.188793
           BTC-USD -0.144042  0.272122  0.190751

[19952 rows x 3 columns]

## 4. Exercises
### Exercise 1 – Hyperparameter Tuning
Vary hidden sizes/dropout and log validation loss per configuration.
<details><summary>Hint</summary>
Wrap the training loop in a function that accepts hidden units and dropout rate.
</details>

### Exercise 2 – GPU Check
Add a cell that uses <code>torch.cuda.is_available()</code> and moves tensors to GPU when available.
<details><summary>Hint</summary>
Call <code>to(device)</code> on both model and tensors.
</details>

### Exercise 3 – Factor Interpretation
Correlate latent factors with the original assets to label them (e.g., growth vs. defensive).
<details><summary>Hint</summary>
Convert <code>latent_factors.numpy()</code> to pandas and compute correlations with feature columns.
</details>


## 5. Takeaways for Chapter 12
- PyTorch lets us prototype deep models inside research notebooks.
- Autoencoders offer a data-driven angle on factor discovery.
- Even simple loops benefit from checkpoints, logging, and validation monitoring.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">